# BERT：从 Encoder-only 原理到下游任务

> **本章定位**：围绕 Encoder-only（仅编码器）Transformer 建立 BERT 的完整数据路径。BERT 使用双向自注意力建模上下文，适用于表征、分类、抽取与掩码语言建模。

> **章节边界**：本章属于跨方向专题：双向语言模型，以 `21` 的 Tokenizer 契约和 `30` 的 Transformer 组件为基础；自回归生成与 Decoder-only 模型不在本章范围内。

**本章总览**：核心机制先以最小原理实现建立可观察基线，再通过标准库接口验证输入输出与数值语义。内容主线如下：

1. WordPiece 与特殊标记 → `BertTokenizerFast`
2. Token / Position / Segment Embedding → `BertEmbeddings`
3. 多头双向自注意力 → `BertSelfAttention`
4. Transformer Encoder → `BertLayer`
5. MLM + NSP 预训练头 → `BertForPreTraining`
6. 下游分类 → `AutoModelForSequenceClassification`

<!-- diagram:bert-overview -->
BERT 的完整数据流由共享 Encoder 主干和不同任务头组成：

![架构图：BERT 从 WordPiece 输入到共享双向 Encoder 与多任务头的数据流](assets/figures/E30_nlp_bert/bert-overview.svg)

[TikZ 源文件](assets/figures/E30_nlp_bert/bert-overview.tex)


## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 跨方向专题：双向语言模型 |
| 本章定位 | 扩展到 Encoder-only Transformer，理解表征学习和 BERT 任务头。 |
| 先修知识 | 掌握 `21` 的 Tokenizer 契约和 `30` 的 Transformer 组件。 |
| 预计时间 | 90～120 分钟 |
| 运行资源 | CPU/Colab；预训练权重按需下载。 |
| 输入 | WordPiece Token、双向 Attention Mask 与 MLM/NSP Labels。 |
| 交付物 | 最小 BERT 预训练模型、标准库对照和下游分类接口。 |

### 1.1．学习目标

完成本章后，读者能够解释 WordPiece、三类 Embedding、双向注意力、Encoder Layer、MLM/NSP 任务头与下游分类接口之间的连续数据流，并能使用形状、Mask 行为和权重共享关系验证实现。


### 1.2．环境与依赖

本章在 Python 3.12.13、PyTorch 2.11.0 与 Transformers 5.13.1 环境下验证。预训练权重按需从 Hugging Face 下载并缓存；原理实现路径不依赖远程权重。


In [ ]:
# %pip install -U "torch>=2.11.0" "transformers>=5.13.1" "datasets>=4.0.0" "accelerate>=1.4.0" "matplotlib>=3.10.0"


In [ ]:
# 固定随机种子并选择可用设备，为后续张量示例建立统一环境。

import math
import random

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

# seed=42 只固定数据顺序与参数初始化；正式结论采用预注册多 Seed 均值和方差，且不承诺跨版本或硬件逐 bit 一致。
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 选择当前可用设备，后续张量和模型统一放到同一计算后端。
DEVICE = torch.accelerator.current_accelerator(check_available=True) or torch.device("cpu")
print("PyTorch:", torch.__version__, "| device:", DEVICE)


## 2．直觉与输入输出契约

BERT 将文本或句对转换为固定词表中的 Token ID，并叠加位置与句段表示；双向 Encoder 产生上下文表示，再由预训练或下游任务头输出 Token 级或序列级预测。

| 阶段 | 输入 | 输出 | 关键约束 |
|---|---|---|---|
| WordPiece | 原始文本或句对 | `input_ids` | 词表、特殊 Token 与规范化规则一致 |
| Embedding | Token、位置、句段 ID | `[B, T, D]` | 三类 Embedding 维度一致 |
| 双向 Encoder | Hidden States、Padding Mask | `[B, T, D]` | 无因果 Mask，仅屏蔽 Padding |
| MLM/NSP | Encoder 输出与监督标签 | Token/句对 Logits | `-100` 位置不参与 MLM 损失 |
| 序列分类 | `[CLS]` 表示 | `[B, C]` | 标签映射与分类头版本化 |


<!-- theory-math-contract:v1 -->
### 2.1．核心机制的语言与数学表达

BERT 使用双向 Attention 建模上下文，并只在被选中的 Mask 位置计算 MLM 损失：

$$
H^{(0)}=E_{\mathrm{token}}+E_{\mathrm{position}}+E_{\mathrm{segment}},\qquad
\mathcal L_{\mathrm{MLM}}=-\frac{1}{|\mathcal M|}\sum_{t\in\mathcal M}\log p_\theta(x_t\mid x_{\setminus\mathcal M})
$$

其中，$H^{(0)}\in\mathbb{R}^{B\times L\times D}$，$\mathcal M$ 是参与预测的位置集合，$x_{\setminus\mathcal M}$ 表示经过 Mask 处理的上下文。`BertEmbeddings` 对应三类 Embedding 求和，`labels == -100` 对应不参与 MLM 损失的位置。双向可见性适合编码任务，但不能直接作为严格自回归生成的因果分解。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

### 3.1．WordPiece 文本编码

BERT 不直接读取字符串，而是把文本变为 token id。WordPiece 从词首开始寻找词表中的最长片段；非词首片段以 `##` 开头。真实 tokenizer 还包含 Unicode 规范化、BasicTokenizer、标点/中文切分、截断、补齐和高性能 Rust 后端。

#### 3.1.1．最长匹配的从零实现


In [ ]:
# 从当前位置选择词表中的最长匹配项，从零实现 WordPiece 贪心切分。

# ID 0…9 是十项手工词表协议；改动词表时必须同步 Tokenizer 与 Embedding 大小。
vocab = {
    "[PAD]": 0, "[UNK]": 1, "[CLS]": 2, "[SEP]": 3, "[MASK]": 4,
    "play": 5, "##ing": 6, "with": 7, "transform": 8, "##ers": 9,
}

def my_wordpiece(word: str, vocab: dict[str, int]) -> list[str]:
    """使用最长优先策略把单词切分为 WordPiece 子词。"""
    word = word.lower()
    tokens = []
    start = 0
    # 重复执行当前步骤，直到达到停止条件或生成完成。
    while start < len(word):
        end = len(word)
        piece = None
        # 重复执行当前步骤，直到达到停止条件或生成完成。
        while start < end:
            candidate = word[start:end]
            if start > 0:
                candidate = "##" + candidate
            if candidate in vocab:
                piece = candidate
                break
            end -= 1
        if piece is None:
            return ["[UNK]"]
        tokens.append(piece)
        start = end
    return tokens

def my_encode(text: str, vocab: dict[str, int]) -> dict[str, list[int]]:
    """编码文本并返回 BERT 所需的 Token、ID、句段和注意力掩码。"""
    pieces = ["[CLS]"]
    for word in text.split():
        pieces.extend(my_wordpiece(word, vocab))
    pieces.append("[SEP]")
    # token_type_id 的 0/1 表示句段 A/B；attention_mask 的 1/0 分别表示有效与 Padding。
    return {
        "tokens": pieces,
        "input_ids": [vocab.get(piece, vocab["[UNK]"]) for piece in pieces],
        "token_type_ids": [0] * len(pieces),
        "attention_mask": [1] * len(pieces),
    }

encoded = my_encode("playing with transformers", vocab)
encoded


#### 3.1.2．`BertTokenizerFast` 接口对照

生产代码通过标准 Tokenizer 统一处理分词细节。`return_tensors="pt"` 直接返回模型需要的张量。


In [ ]:
# 把同一词表交给 BertTokenizerFast，切换到标准分词接口。

from transformers import BertTokenizerFast

# 初始化 Tokenizer，并固定文本与 token ID 之间的转换协议。
lib_tokenizer = BertTokenizerFast.from_pretrained("google-bert/bert-base-uncased")
lib_batch = lib_tokenizer(
    ["playing with transformers", "BERT understands both sides"],
    padding=True,
    truncation=True,
    return_tensors="pt",
)
print(lib_tokenizer.convert_ids_to_tokens(lib_batch["input_ids"][0]))
{lib_key: tuple(lib_value.shape) for lib_key, lib_value in lib_batch.items()}


### 3.2．三类 Embedding 的组合

对位置 $i$ 的输入：

$$x_i = E_{token}(id_i) + E_{position}(i) + E_{segment}(type_i)$$

BERT 使用可学习的绝对位置向量；segment embedding 区分句子 A/B。相加后再做 LayerNorm 和 Dropout。

#### 3.2.1．BERT Embedding 的从零实现


In [ ]:
# 相加 Token、Position 与 Segment Embedding，再归一化得到 BERT 输入。

class MyBertEmbeddings(nn.Module):
    """融合 Token、位置和句段 Embedding，形成 BERT 输入表示。"""
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    # dropout=0.1 是经典 BERT 训练起点；加载预训练权重时必须服从 Config。
    def __init__(self, vocab_size: int, hidden_size: int, max_length: int, dropout: float = 0.1):
        """创建三类 Embedding、LayerNorm 与 Dropout。"""
        super().__init__()
        self.token = nn.Embedding(vocab_size, hidden_size, padding_idx=0)
        self.position = nn.Embedding(max_length, hidden_size)
        self.segment = nn.Embedding(2, hidden_size)
        # epsilon=1e-12 防止方差为零时除零，并保持经典 BERT 数值契约。
        self.norm = nn.LayerNorm(hidden_size, eps=1e-12)
        self.dropout = nn.Dropout(dropout)

    # 前向传播按照本模块的数据流连接各子层，并返回当前阶段输出。
    def forward(self, input_ids: torch.Tensor, token_type_ids: torch.Tensor | None = None):
        """返回与输入 ID 同序列长度的归一化 BERT Embedding。"""
        batch_size, seq_len = input_ids.shape
        if token_type_ids is None:
            token_type_ids = torch.zeros_like(input_ids)
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
        hidden = (
            self.token(input_ids)
            + self.position(positions)
            + self.segment(token_type_ids)
        )
        return self.dropout(self.norm(hidden))

# hidden=32 与 max positions=32 控制夹具规模；dropout=0.0 排除随机掩码，真实容量与正则化服从 Config。
embedding = MyBertEmbeddings(vocab_size=10, hidden_size=32, max_length=32, dropout=0.0)
ids = torch.tensor([encoded["input_ids"]])
types = torch.tensor([encoded["token_type_ids"]])
hidden = embedding(ids, types)
print(hidden.shape)


#### 3.2.2．`BertEmbeddings` 接口对照

这是 Hugging Face BERT 内部使用的正式实现。生产代码通常不直接导入内部类，而是通过 `BertModel` 调用；本节单独调用该组件，以建立原理实现与库对象的一一对应关系。


In [ ]:
# 使用 BertEmbeddings 复现相同的三类 Embedding 组合流程。

from transformers import BertConfig
from transformers.models.bert.modeling_bert import BertEmbeddings

# 集中定义结构和运行参数，避免配置散落在后续逻辑中。
lib_config = BertConfig(vocab_size=10, hidden_size=32, max_position_embeddings=32, hidden_dropout_prob=0.0)
lib_embedding = BertEmbeddings(lib_config)
lib_hidden = lib_embedding(input_ids=ids, token_type_ids=types)
print(lib_hidden.shape)


### 3.3．双向多头自注意力

单头注意力为：

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^{\top}}{\sqrt{d_h}} + M_{\mathrm{pad}}\right)V$$

BERT 没有因果 mask，因此一个 token 能同时看左侧和右侧；padding mask 只屏蔽补齐位置。多头机制让不同子空间学习不同关系。

#### 3.3.1．多头自注意力的从零实现


In [ ]:
# 将隐藏维拆为多个头，计算全双向自注意力后再合并。

class MyMultiHeadSelfAttention(nn.Module):
    """实现支持 Key Padding Mask 的双向多头自注意力。"""
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    def __init__(self, hidden_size: int, num_heads: int, dropout: float = 0.0):
        """按隐藏宽度和头数创建 Q/K/V/O 投影。"""
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.q = nn.Linear(hidden_size, hidden_size)
        self.k = nn.Linear(hidden_size, hidden_size)
        self.v = nn.Linear(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def my_split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """把 `[B,S,H]` 表示重排为 `[B,heads,S,head_dim]`。"""
        b, s, hidden_size = x.shape
        # [B, S, H] → [B, heads, S, head_dim]。
        return x.view(b, s, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor | None = None):
        """计算双向自注意力上下文并返回逐头权重。"""
        q = self.my_split_heads(self.q(x))
        k = self.my_split_heads(self.k(x))
        v = self.my_split_heads(self.v(x))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        if attention_mask is not None:
            # 扩展为可广播到所有注意力头和 Query 位置的 Key mask。
            key_mask = attention_mask[:, None, None, :].bool()
            scores = scores.masked_fill(~key_mask, torch.finfo(scores.dtype).min)
        weights = self.dropout(scores.softmax(dim=-1))
        context = weights @ v
        b, num_heads, s, head_dim = context.shape
        context = context.transpose(1, 2).contiguous().view(b, s, -1)
        return self.out(context), weights

# 4 个 Head 可整除 hidden=32；增加宽度或 Head 数会提高参数、内存与延迟。
attention = MyMultiHeadSelfAttention(hidden_size=32, num_heads=4)
mask = torch.ones(hidden.shape[:2], dtype=torch.long)
context, weights = attention(hidden, mask)
print("context:", context.shape, "attention:", weights.shape)


#### 3.3.2．`BertSelfAttention` 接口对照

Hugging Face 的 mask 是加到注意力分数上的 4D additive mask：保留位置为 0，屏蔽位置为一个很大的负数。


In [ ]:
# 构造 BertSelfAttention，观察标准库的输入输出张量契约。

from transformers.models.bert.modeling_bert import BertSelfAttention

# 集中定义结构和运行参数，避免配置散落在后续逻辑中。
lib_attn_config = BertConfig(hidden_size=32, num_attention_heads=4, attention_probs_dropout_prob=0.0)
lib_attention = BertSelfAttention(lib_attn_config)
lib_additive_mask = (1.0 - mask[:, None, None, :].float()) * torch.finfo(torch.float32).min
lib_context, lib_weights = lib_attention(
    hidden_states=hidden,
    attention_mask=lib_additive_mask,
    output_attentions=True,
)[:2]
print("context:", lib_context.shape, "attention:", lib_weights.shape)


### 3.4．Encoder Layer：Attention、FFN 与残差

BERT 的每层包含注意力子层和逐 token 的前馈网络。原始 BERT 采用 Post-LayerNorm：先残差相加，再 LayerNorm。前馈层通常把维度扩张 4 倍并使用 GELU。

#### 3.4.1．Encoder Layer 的从零实现

<!-- diagram:bert-encoder-layer -->
BERT Encoder Layer 使用两次残差连接，把注意力和逐位置 FFN 串联起来：

![架构图：BERT Encoder Layer 的两条 Post-LayerNorm 残差子层](assets/figures/E30_nlp_bert/bert-encoder-layer.svg)

[TikZ 源文件](assets/figures/E30_nlp_bert/bert-encoder-layer.tex)


In [ ]:
# 组合自注意力、残差归一化与前馈网络，形成最小 BERT Encoder Layer。

class MyBertLayer(nn.Module):
    """组合 Post-Norm 自注意力与前馈网络的 BERT Encoder Layer。"""
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    def __init__(self, hidden_size: int, num_heads: int, intermediate_size: int, dropout: float = 0.1):
        """创建注意力、两层 FFN、残差归一化与 Dropout。"""
        super().__init__()
        self.attention = MyMultiHeadSelfAttention(hidden_size, num_heads, dropout)
        self.attn_norm = nn.LayerNorm(hidden_size, eps=1e-12)
        self.ffn_in = nn.Linear(hidden_size, intermediate_size)
        self.ffn_out = nn.Linear(intermediate_size, hidden_size)
        self.ffn_norm = nn.LayerNorm(hidden_size, eps=1e-12)
        self.dropout = nn.Dropout(dropout)

    # 前向传播按照本模块的数据流连接各子层，并返回当前阶段输出。
    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor | None = None):
        """依次执行注意力和 FFN 残差更新并返回注意力权重。"""
        attn, weights = self.attention(x, attention_mask)
        x = self.attn_norm(x + self.dropout(attn))
        ffn = self.ffn_out(F.gelu(self.ffn_in(x)))
        x = self.ffn_norm(x + self.dropout(ffn))
        return x, weights

# FFN=128 是 hidden=32 的 4 倍；本次对照关闭 dropout 以保持数值路径确定。
layer = MyBertLayer(32, 4, 128, dropout=0.0)
layer_output, layer_weights = layer(hidden, mask)
print(layer_output.shape)


#### 3.4.2．`BertLayer` 接口对照


In [ ]:
from transformers.models.bert.modeling_bert import BertLayer

# 集中定义结构和运行参数，避免配置散落在后续逻辑中。
lib_layer_config = BertConfig(
    hidden_size=32,
    num_attention_heads=4,
    intermediate_size=128,
    hidden_dropout_prob=0.0,
    attention_probs_dropout_prob=0.0,
)
# 直接实例化内部层时需显式选择后端；BertModel.from_pretrained 会自动完成该配置。
lib_layer_config._attn_implementation = "eager"
lib_layer = BertLayer(lib_layer_config)
lib_layer_result = lib_layer(hidden, attention_mask=lib_additive_mask)
# Transformers 5.x 直接返回 Tensor；较早版本可能返回 tuple。
lib_layer_output = lib_layer_result[0] if isinstance(lib_layer_result, tuple) else lib_layer_result
print(lib_layer_output.shape)


### 3.5．预训练目标：MLM 与 NSP

- **MLM**：经典数据配方选择约 `15%` Token，其中通常 `80%` 替换为 `[MASK]`、`10%` 替换为随机 Token、`10%` 保持原值；它不是当前网络常数，动态或 span masking 需重新评测，验证集 Mask 必须可复现。只在被选中的 Token 位置计算交叉熵，标签为 `-100` 的位置被忽略。
- **NSP**：经典配方使用约 `50/50` 的正负句对以平衡二分类，并用 `[CLS]` 表示预测句子 B 是否紧随句子 A；后续模型常移除 NSP，不能将该比例套用于无 NSP 的模型。

#### 3.5.1．BERT 预训练闭环的最小实现

<!-- diagram:bert-pretraining-heads -->
MLM 与 NSP 共享同一个 BERT 主干，但分别提供 token 级和句对级监督：

![架构图：BERT 共享 Encoder 上的 MLM Token 级监督与 NSP 句对级监督](assets/figures/E30_nlp_bert/bert-pretraining-heads.svg)

[TikZ 源文件](assets/figures/E30_nlp_bert/bert-pretraining-heads.tex)


In [ ]:
# 堆叠 Encoder，并让 MLM Decoder 与 Token Embedding 共享权重。

class MyBertForPreTraining(nn.Module):
    """堆叠 BERT Encoder，并提供权重绑定的 MLM 与 NSP 任务头。"""
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    def __init__(self, vocab_size: int, hidden_size: int, num_heads: int, num_layers: int):
        """按指定层数创建 Embedding、Encoder 和两个预训练头。"""
        super().__init__()
        self.embeddings = MyBertEmbeddings(vocab_size, hidden_size, 32)
        self.layers = nn.ModuleList([
            MyBertLayer(hidden_size, num_heads, hidden_size * 4)
            for layer_index in range(num_layers)
        ])
        self.mlm_transform = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.LayerNorm(hidden_size, eps=1e-12),
        )
        self.mlm_decoder = nn.Linear(hidden_size, vocab_size, bias=False)
        self.mlm_decoder.weight = self.embeddings.token.weight
        self.mlm_bias = nn.Parameter(torch.zeros(vocab_size))
        self.nsp = nn.Linear(hidden_size, 2)

    # 前向传播按照本模块的数据流连接各子层，并返回当前阶段输出。
    def forward(self, input_ids, attention_mask, token_type_ids=None):
        """返回每个位置的 MLM logits 与 CLS 位置的 NSP logits。"""
        x = self.embeddings(input_ids, token_type_ids)
        for layer in self.layers:
            x, unused_attention = layer(x, attention_mask)
        mlm_logits = self.mlm_decoder(self.mlm_transform(x)) + self.mlm_bias
        nsp_logits = self.nsp(x[:, 0])
        return mlm_logits, nsp_logits

# 两层 Encoder 足以验证堆叠与双任务头；层数增加会同步提高计算、内存与延迟。
model = MyBertForPreTraining(vocab_size=10, hidden_size=32, num_heads=4, num_layers=2)
mlm_logits, nsp_logits = model(ids, mask, types)
print("MLM:", mlm_logits.shape, "NSP:", nsp_logits.shape)


#### 3.5.2．`BertForPreTraining` 接口对照

生产中应直接加载经过大规模预训练的权重，而不是从随机初始化开始。该接口同时返回 MLM 和 NSP logits。


In [ ]:
# 用 BertForPreTraining 接管 MLM 与 NSP 头，切换到官方模型结构。

from transformers import BertForPreTraining

lib_pretrain_model = BertForPreTraining.from_pretrained("google-bert/bert-base-uncased").to(DEVICE)
lib_inputs = {lib_key: lib_value.to(DEVICE) for lib_key, lib_value in lib_batch.items()}
lib_pretrain_model.eval()
# 关闭梯度记录，避免推理或参数更新阶段构建额外计算图。
with torch.inference_mode():
    lib_output = lib_pretrain_model(**lib_inputs)
print("MLM:", lib_output.prediction_logits.shape, "NSP:", lib_output.seq_relationship_logits.shape)


### 3.6．下游序列分类

`AutoModelForSequenceClassification` 会选择与 checkpoint 匹配的模型类，并在 `[CLS]` 表示上增加分类头。本节使用两条样本展示训练接口；实际项目应使用 `datasets`、验证集、动态 padding、学习率调度、梯度裁剪、混合精度和指标追踪。


In [ ]:
# 加载序列分类模型与配套 Tokenizer，展示真实下游任务入口。

from transformers import AutoModelForSequenceClassification, AutoTokenizer

lib_cls_tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
lib_cls_model = AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-base-uncased", num_labels=2
).to(DEVICE)
lib_train_batch = lib_cls_tokenizer(
    ["this course is clear", "the result is disappointing"],
    padding=True,
    truncation=True,
    return_tensors="pt",
)
lib_train_batch = {lib_key: lib_value.to(DEVICE) for lib_key, lib_value in lib_train_batch.items()}
lib_labels = torch.tensor([1, 0], device=DEVICE)
# lr=2e-5、weight_decay=0.01 是预训练 BERT 微调起点；需结合任务、有效 batch、warmup 与训练预算重调。
# betas=(0.9,0.999)、eps=1e-8 与学习率共同定义更新，生产配置应显式落盘。
lib_optimizer = torch.optim.AdamW(
    lib_cls_model.parameters(),
    lr=2e-5, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01,
)

lib_cls_model.train()
# 清空上一轮梯度，避免 PyTorch 默认的梯度累积。
lib_optimizer.zero_grad(set_to_none=True)
lib_loss = lib_cls_model(**lib_train_batch, labels=lib_labels).loss
# 反向传播计算梯度，供随后的参数更新使用。
lib_loss.backward()
# 裁剪阈值 1.0 用于限制本次更新；若频繁触发，应先排查学习率与数值稳定性。
torch.nn.utils.clip_grad_norm_(lib_cls_model.parameters(), 1.0)
lib_optimizer.step()
print("one-step loss:", float(lib_loss.detach()))


## 4．证据验证

1. 将 `mask` 的最后一个位置设为 0，各注意力头对该 Key 的权重应接近 0。
2. MLM Decoder 与 Token Embedding 的共享权重应指向同一参数，并保持梯度可传播。
3. BERT 双向 Mask 与 GPT Causal Mask 在未来位置的可见性上应呈现明确差异。
4. 原理实现与库接口的 Hidden States、MLM Logits、NSP Logits 和分类 Logits 形状应一致。
5. 微调产物应同时保存 Tokenizer、Config、权重与标签映射，并能在独立进程中恢复。


### 4.1．双向 Attention 与 Causal Mask 对照

**学习问题。** 在 Q、K 投影和输入 Hidden States 完全相同的条件下，BERT 双向可见性与 GPT 式因果可见性会怎样改变注意力矩阵？未来位置的权重差异能否明确归因于 Mask，而不是参数或输入变化？

本图直接复用 `hidden`、`mask`、`attention.my_split_heads()`、`attention.q/k` 以及前文已经得到的 `weights`。代码先重建同一份缩放点积分数，并核对未加 causal mask 的结果与 `MyMultiHeadSelfAttention.forward()` 输出一致；随后只在同一分数矩阵上增加上三角 causal mask。图中展示 Head 0，数值检查覆盖全部 4 个 heads。

运行前应明确以下形状与数值不变量：

- 两组权重形状都为 `[B, heads, T, T] = [1, 4, 7, 7]`；两个 `T` 依次表示 Query 与 Key 位置。
- 两组权重均非负，每个 `[batch, head, query]` 行沿 Key 维求和为 1。当前 `mask` 全为 1，因此没有 Padding Key。
- BERT 双向权重只应用 Padding Mask，主对角线上方的未来 Key 可以获得非零概率；Causal 对照在同一 scores 上把 `key_position > query_position` 置为极小值，softmax 后未来权重应为 0。
- 未加 causal mask 的重建结果必须与现有 `weights` 在浮点容差内一致，从而把两图差异限定为 Mask 语义。


In [ ]:
# 固定同一组 Q/K 分数，只切换双向与因果可见性。
import matplotlib.pyplot as plt

with torch.no_grad():
    comparison_q = attention.my_split_heads(attention.q(hidden))
    comparison_k = attention.my_split_heads(attention.k(hidden))
    shared_scores = (
        comparison_q @ comparison_k.transpose(-2, -1)
        / math.sqrt(attention.head_dim)
    )
    comparison_key_mask = mask[:, None, None, :].bool()
    bidirectional_scores = shared_scores.masked_fill(
        ~comparison_key_mask, torch.finfo(shared_scores.dtype).min
    )
    rebuilt_bidirectional_weights = bidirectional_scores.softmax(dim=-1)
    sequence_length = hidden.size(1)
    causal_future_mask = torch.triu(
        torch.ones(
            sequence_length, sequence_length,
            dtype=torch.bool, device=hidden.device,
        ),
        diagonal=1,
    )
    causal_scores = bidirectional_scores.masked_fill(
        causal_future_mask[None, None, :, :],
        torch.finfo(shared_scores.dtype).min,
    )
    causal_weights = causal_scores.softmax(dim=-1)

bidirectional_weights = weights.detach()
expected_shape = (hidden.size(0), attention.num_heads, sequence_length, sequence_length)
if tuple(bidirectional_weights.shape) != expected_shape or tuple(causal_weights.shape) != expected_shape:
    raise RuntimeError("双向与因果注意力形状未保持 [B, heads, T, T]")
if not torch.allclose(
    bidirectional_weights, rebuilt_bidirectional_weights, atol=1e-6, rtol=1e-5
):
    raise RuntimeError("重建的双向权重与 MyMultiHeadSelfAttention 输出不一致")
bidirectional_row_error = (
    bidirectional_weights.sum(dim=-1) - 1.0
).abs().max().item()
causal_row_error = (causal_weights.sum(dim=-1) - 1.0).abs().max().item()
causal_future_max = causal_weights[:, :, causal_future_mask].abs().max().item()
bidirectional_future_mass = (
    bidirectional_weights
    * causal_future_mask[None, None, :, :].to(dtype=bidirectional_weights.dtype)
).sum(dim=-1)
if bidirectional_weights.min().item() < 0.0 or causal_weights.min().item() < 0.0:
    raise RuntimeError("softmax 后出现负注意力权重")
if bidirectional_row_error > 1e-6 or causal_row_error > 1e-6:
    raise RuntimeError("注意力权重没有沿 Key 维归一化")
if causal_future_max > 1e-7:
    raise RuntimeError("Causal Mask 未把未来位置权重压到零")
if bidirectional_future_mass.max().item() <= 0.0:
    raise RuntimeError("全有效输入上的 BERT 双向注意力没有读取任何未来位置")

token_labels = encoded["tokens"]
if len(token_labels) != sequence_length:
    raise RuntimeError("可视化 Token 标签与注意力序列长度不一致")
head_index = 0
matrices = [
    ("BERT：双向 Attention", bidirectional_weights[0, head_index].detach().cpu()),
    ("GPT 对照：Causal Mask", causal_weights[0, head_index].detach().cpu()),
]
common_max = max(matrix.max().item() for _, matrix in matrices)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.8), constrained_layout=True)
for ax, (title, matrix) in zip(axes, matrices):
    image = ax.imshow(
        matrix, cmap="viridis", vmin=0.0, vmax=common_max,
        interpolation="nearest", aspect="equal",
    )
    for query_index in range(sequence_length):
        for key_index in range(sequence_length):
            value = float(matrix[query_index, key_index])
            ax.text(
                key_index, query_index, f"{value:.2f}",
                ha="center", va="center", fontsize=7,
                color="white" if value > common_max * 0.52 else "#0f172a",
            )
    ax.set_xticks(range(sequence_length), token_labels, rotation=45, ha="right")
    ax.set_yticks(range(sequence_length), token_labels)
    ax.set_xlabel("Key：被读取位置")
    ax.set_ylabel("Query：当前位置")
    ax.set_title(title)
fig.colorbar(image, ax=axes.tolist(), shrink=0.82, label="注意力权重")
fig.suptitle("同一 Hidden States、Q/K 参数与 scores：只改变 Mask")
plt.show()

print(
    "双向/因果注意力契约：",
    {
        "shape": expected_shape,
        "bidirectional_max_row_sum_error": bidirectional_row_error,
        "causal_max_row_sum_error": causal_row_error,
        "causal_max_future_weight": causal_future_max,
        "bidirectional_mean_future_mass": bidirectional_future_mass.mean().item(),
    },
)


**应观察到的结论。** BERT 面板的主对角线上方存在非零权重，说明每个非末尾位置能够读取右侧上下文；Causal 面板的同一区域全部为零。由于两组结果共享输入、Q/K 投影和缩放 scores，允许区域内权重的变化来自屏蔽未来位置后的再次归一化，而不是模型参数变化。

**不可误读的边界。** 这些权重来自随机初始化的微型层，只证明双向与因果 Mask 的信息流语义，不代表学到了句法、语义或可解释的重要性，也不能据此比较 BERT 与 GPT 的任务质量。当前夹具没有 Padding，因而本图未验证 Padding Key 的屏蔽；相关检查仍需单独把 `mask` 的末位设为 0。BERT 在 MLM 中能够读取目标两侧上下文，但被预测内容需要先由数据构造替换或遮盖，双向 Attention 并不意味着可以直接复制未遮盖答案，也不使 Encoder-only 模型具备自回归生成契约。


## 5．迁移到生产库

| 原理对象 | 生产库对象 | 迁移重点 |
|---|---|---|
| WordPiece 最长匹配 | `BertTokenizerFast` / `AutoTokenizer` | 规范化、特殊 Token、截断与 Padding |
| 三类 Embedding | `BertModel` 内部 Embedding | Config、位置上限与句段 ID |
| 双向注意力与 Encoder Layer | `BertLayer` / `BertModel` | Additive Mask、Dropout 与输出对象 |
| MLM + NSP | `BertForPreTraining` | 权重共享、Label 协议与 Loss Reduction |
| 序列分类 | `AutoModelForSequenceClassification` | 标签映射、分类头与保存加载 |

生产训练以预训练检查点及其配套 Tokenizer 为起点，原理实现用于公式验证与回归基线，不承担模型训练和服务执行。


## 6．生产边界

1. Tokenizer、词表、特殊 Token、模型 Config 和权重应来自同一模型 ID；加载后校验关键文件哈希与接口契约，避免词表错配造成静默语义错误。
2. 长文本策略需要明确截断、滑窗或分段规则，并分别记录截断率和任务指标。
3. 微调数据应划分训练、验证与测试集合，动态 Padding、混合精度、梯度裁剪和指标追踪由成熟训练框架负责。
4. Padding Mask 方向、`token_type_ids` 语义及训练/推理 Dropout 状态需要纳入回归测试。
5. 模型产物应包含标签映射、依赖版本、评测证据和恢复路径。
